# Clase 065 — Precision/Recall tradeoff

No se puede maximizar precision y recall a la vez: mover el **umbral** de decisión sube uno y baja el otro. Usamos `decision_function` + `precision_recall_curve` para elegir el umbral según el costo del negocio, no según el default `0`.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import (precision_recall_curve, precision_score,
                             recall_score, average_precision_score)

np.random.seed(42)

digits = load_digits()
X, y = digits.data, digits.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
ytr5 = (ytr == 5)
print('positivos (5) en train:', int(ytr5.sum()), '/', len(ytr5))

## 1. Score crudo vs clase

`decision_function` da el score continuo; `predict` ya aplicó el umbral 0. Verificamos que `predict` equivale a `score > 0`.

In [ ]:
sgd = SGDClassifier(random_state=42)
sgd.fit(Xtr, ytr5)

scores0 = sgd.decision_function(Xtr[:5])
preds0 = sgd.predict(Xtr[:5])
for s, p in zip(scores0, preds0):
    print(f'score={s:8.3f} -> predict={bool(p)}  (score>0 = {bool(s > 0)})')

assert np.array_equal(preds0, scores0 > 0)

## 2. Curva precision/recall vs umbral

Con `cross_val_predict(method='decision_function')` obtenemos scores out-of-fold (sin leakage) y los pasamos a `precision_recall_curve`. Ojo: precision/recall tienen un elemento más que thresholds.

In [ ]:
y_scores = cross_val_predict(sgd, Xtr, ytr5, cv=3,
                             method='decision_function', n_jobs=1)
precisions, recalls, thresholds = precision_recall_curve(ytr5, y_scores)
print('precisions:', precisions.shape, '| thresholds:', thresholds.shape)
assert len(precisions) == len(thresholds) + 1

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, precisions[:-1], label='precision', color='#37a')
ax.plot(thresholds, recalls[:-1], label='recall', color='#c33')
ax.set_xlabel('umbral (threshold)')
ax.set_ylabel('valor')
ax.set_title('Precision y recall vs umbral')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Umbral para precision ≥ 90%

Buscamos el umbral mínimo que garantiza `precision >= 0.90` y verificamos la precision resultante al aplicarlo sobre los scores.

In [ ]:
objetivo = 0.90
idx = np.argmax(precisions >= objetivo)
umbral_90 = thresholds[idx]
y_pred_90 = (y_scores >= umbral_90)

prec_90 = precision_score(ytr5, y_pred_90)
rec_90 = recall_score(ytr5, y_pred_90)
print(f'umbral elegido: {umbral_90:.3f}')
print(f'precision resultante: {prec_90:.3f}')
print(f'recall resultante:    {rec_90:.3f}')

assert prec_90 >= objetivo

## 4. Curva precision vs recall

La forma canónica de la curva PR: precision (Y) contra recall (X). Marcamos el punto del umbral por default (0) y el del umbral de precision ≥ 90%.

In [ ]:
prec_def = precision_score(ytr5, y_scores > 0)
rec_def = recall_score(ytr5, y_scores > 0)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recalls, precisions, color='#37a')
ax.scatter([rec_def], [prec_def], color='black', zorder=5, label='umbral=0 (default)')
ax.scatter([rec_90], [prec_90], color='#c33', zorder=5, label='precision>=0.90')
ax.set_xlabel('recall')
ax.set_ylabel('precision')
ax.set_title('Curva Precision-Recall')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Average precision (AP)

`average_precision_score` resume toda la curva en un escalar (área bajo la curva PR). Es más informativo que un F1 a umbral fijo cuando hay desbalanceo.

In [ ]:
ap = average_precision_score(ytr5, y_scores)
print(f'average precision (AP): {ap:.4f}')

from sklearn.metrics import f1_score
f1_def = f1_score(ytr5, y_scores > 0)
print(f'F1 al umbral default:   {f1_def:.4f}')
print('\nAP integra sobre todos los umbrales; F1 es a un umbral fijo.')

assert 0.0 <= ap <= 1.0 and ap > 0.5

## Ejercicios

1. **Umbral para recall ≥ 95%.** Igual que el punto 3 pero fijando recall. ¿Qué precision conseguís?
2. **predict_proba.** Repetí con `SGDClassifier(loss='log_loss')` y `predict_proba(X)[:, 1]` como score. ¿Cambia la curva?
3. **Comparar métricas.** ¿AP o F1 te parece más informativo para elegir modelo? Justificá con los números obtenidos.
4. **Otro clasificador.** `RandomForestClassifier` no tiene `decision_function`: usá `predict_proba(X)[:, 1]` y compará su AP contra el SGD.

## Conclusiones

- El umbral `0` de `predict` **no es sagrado**: define cuántos FP y FN tolerás.
- `decision_function` (o `predict_proba`) da el score continuo necesario para mover el umbral.
- `precision_recall_curve` devuelve `len(thresholds)+1` precisions/recalls: graficá `[:-1]` vs thresholds.
- Elegí el umbral con scores **out-of-fold** (`cross_val_predict`) para no sobreajustarlo.
- `average_precision_score` resume la curva PR; brilla en datasets desbalanceados.

## ✅ Soluciones de los ejercicios

Ejercicios del README resueltos sobre el detector 'es un 5' de `load_digits` (offline). Reutilizamos `sgd`, `ytr5`, `y_scores`, `precisions`, `recalls` y `thresholds` calculados arriba.

**Ej. 1 — Score crudo.** `decision_function` da el score continuo; `predict` ya aplicó el umbral 0. Mostramos que `predict` equivale a `score > 0`.

In [ ]:

import numpy as np
s = sgd.decision_function(Xtr[:1])[0]
p = bool(sgd.predict(Xtr[:1])[0])
print(f"decision_function(X[0]) = {s:.3f} | predict = {p} | (score>0) = {bool(s > 0)}")
# sobre un lote:
lote = sgd.decision_function(Xtr[:20])
assert np.array_equal(sgd.predict(Xtr[:20]), lote > 0)
print("OK: predict es exactamente decision_function > 0")

**Ej. 2 — Curva PR vs umbral.** Con scores out-of-fold (`cross_val_predict`) graficamos precision y recall en función del threshold. Al subir el umbral, sube precision y baja recall.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve

ys = cross_val_predict(sgd, Xtr, ytr5, cv=3, method="decision_function", n_jobs=1)
prec, rec, ths = precision_recall_curve(ytr5, ys)
assert len(prec) == len(ths) + 1
plt.plot(ths, prec[:-1], label="precision"); plt.plot(ths, rec[:-1], label="recall")
plt.xlabel("threshold"); plt.ylabel("valor"); plt.legend(); plt.title("Precision y recall vs umbral"); plt.show()
print("al mover el umbral a la derecha: precision sube, recall baja (no se maximizan juntas)")

**Ej. 3 — Umbral para precision ≥ 90%.** Buscamos el threshold mínimo que garantiza `precision >= 0.90` y verificamos precision/recall resultantes.

In [ ]:

import numpy as np
from sklearn.metrics import precision_score, recall_score

idx = int(np.argmax(prec >= 0.90))
th90 = ths[idx]
y_pred_90 = (ys >= th90)
print(f"threshold_90 = {th90:.3f}")
print(f"precision={precision_score(ytr5, y_pred_90):.3f} | recall={recall_score(ytr5, y_pred_90):.3f}")
assert precision_score(ytr5, y_pred_90) >= 0.90

**Ej. 4 — Curva precision vs recall.** La forma canónica (precision en Y, recall en X). Marcamos el punto del umbral por default (0).

In [ ]:

import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score

pd_def = precision_score(ytr5, ys > 0); rd_def = recall_score(ytr5, ys > 0)
plt.plot(rec, prec)
plt.scatter([rd_def], [pd_def], color="black", zorder=5, label="umbral=0 (default)")
plt.xlabel("recall"); plt.ylabel("precision"); plt.legend(); plt.title("Curva Precision-Recall"); plt.show()
print(f"punto default: recall={rd_def:.3f}, precision={pd_def:.3f}")

**Ej. 5 — Average precision.** `average_precision_score` resume toda la curva PR en un escalar; con desbalanceo es más informativo que un F1 a umbral fijo.

In [ ]:

from sklearn.metrics import average_precision_score, f1_score

ap = average_precision_score(ytr5, ys)
f1_def = f1_score(ytr5, ys > 0)
print(f"average precision (integra todos los umbrales): {ap:.4f}")
print(f"F1 al umbral default (un solo umbral):          {f1_def:.4f}")
print("AP es mas informativo para comparar modelos: no depende de un umbral arbitrario")
assert 0.5 < ap <= 1.0